In [0]:
#Célula 1 — dependências
%pip install python-dotenv
dbutils.library.restartPython()

In [0]:
#Célula 2 — variáveis de ambiente
import os
from dotenv import load_dotenv

env_path = os.path.join(os.getcwd(), ".env")
print("Procurando .env em:", env_path, "| existe:", os.path.exists(env_path))
load_dotenv(env_path, override=True)

for var in ["ADLS_TENANT_ID", "ADLS_CLIENT_ID", "ADLS_CLIENT_SECRET",
            "ADLS_STORAGE_ACCOUNT", "ADLS_CONTAINER",
            "SQL_HOST", "SQL_DATABASE", "SQL_USERNAME", "SQL_PASSWORD"]:
    print(f"{var}: {'OK' if os.getenv(var) else 'FALTANDO'}")

In [0]:
# Célula 3 — leitura via Spark nativo + metadados de batch
from pyspark.sql import functions as F

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}
base_url = f"abfss://{os.getenv('ADLS_CONTAINER')}@{os.getenv('ADLS_STORAGE_ACCOUNT')}.dfs.core.windows.net"

PADRAO_PASTA = r"real-time-data/(\d{4})/(\d{2})/(\d{2})/(\d{6})/"

def com_metadados_batch(df):
    caminho = F.col("_metadata.file_path")
    return (df
        .withColumn("_ano", F.regexp_extract(caminho, PADRAO_PASTA, 1))
        .withColumn("_mes", F.regexp_extract(caminho, PADRAO_PASTA, 2))
        .withColumn("_dia", F.regexp_extract(caminho, PADRAO_PASTA, 3))
        .withColumn("_hora_batch", F.regexp_extract(caminho, PADRAO_PASTA, 4))
        .withColumn("_batch_id", F.concat_ws("_", F.col("_ano"), F.col("_mes"), F.col("_dia"), F.col("_hora_batch")))
        .withColumn("_reference_date", F.to_date(F.concat_ws("-", F.col("_ano"), F.col("_mes"), F.col("_dia"))))
        .withColumn("_snapshot", F.col("_hora_batch"))
        .drop("_ano", "_mes", "_dia", "_hora_batch"))

df_produtos = com_metadados_batch(
    spark.read.format("parquet").options(**adls_options)
    .load(f"{base_url}/real-time-data/*/*/*/*/ecommerce_produtos.parquet"))

df_categorias = (spark.read.format("parquet").options(**adls_options)
    .load(f"{base_url}/real-time-data/*/*/*/*/ecommerce_categorias.parquet")
    .dropDuplicates(["id_categoria", "nome_categoria", "id_categoria_pai", "nome_categoria_pai", "tipo_categoria"]))

print("produtos:", df_produtos.count(), "| categorias:", df_categorias.count())

In [0]:
#Célula 4 — qualidade dos dados
snaps = sorted(r["_snapshot"] for r in df_produtos.select("_snapshot").distinct().collect())
print("snapshots encontrados:", snaps)

# 1. nulos e duplicidade de chave por snapshot
(df_produtos.groupBy("_snapshot")
    .agg(F.count("*").alias("linhas"),
         F.countDistinct("sku").alias("skus_unicos"),
         *[F.sum(F.col(c).isNull().cast("int")).alias(f"nulos_{c}")
           for c in df_produtos.columns if c not in ("_snapshot",)])
    .orderBy("_snapshot")
    .show(truncate=False))

# 2. produtos existem em todos os snapshots ou mudam? (decide append puro vs. dedup por sku)
primeiro, ultimo = snaps[0], snaps[-1]
sku_primeiro = set(r["sku"] for r in df_produtos.filter(F.col("_snapshot") == primeiro).select("sku").collect())
sku_ultimo = set(r["sku"] for r in df_produtos.filter(F.col("_snapshot") == ultimo).select("sku").collect())
print(f"SKUs em comum entre {primeiro} e {ultimo}: {len(sku_primeiro & sku_ultimo)} de {len(sku_ultimo)}")

# 3. regras de negócio (último snapshot)
ult = df_produtos.filter(F.col("_snapshot") == ultimo)
print("categoria inexistente:", ult.join(df_categorias, "id_categoria", "left_anti").count())
print("preco_lista <= 0:", ult.filter(F.col("preco_lista") <= 0).count())
ult.select(F.min("preco_lista"), F.max("preco_lista"), F.avg("preco_lista")).show()

# 4. categorias: hierarquia
pai_inexistente = (df_categorias.filter(F.col("id_categoria_pai").isNotNull())
    .join(df_categorias.select(F.col("id_categoria").alias("id_categoria_pai")), "id_categoria_pai", "left_anti"))
print("categoria pai inexistente:", pai_inexistente.count())
print("id_categoria duplicado:", df_categorias.count() - df_categorias.dropDuplicates(["id_categoria"]).count())

In [0]:
#Célula 5 — conexão com o SQL Server
def opts_sql():
    return {"host": os.getenv("SQL_HOST"), "port": "1433",
            "database": os.getenv("SQL_DATABASE"),
            "user": os.getenv("SQL_USERNAME"),
            "password": os.getenv("SQL_PASSWORD"), "encrypt": "true"}

# PROVISÓRIO — trocar quando o Geovany confirmar o padrão squad1/nome/ecommerce
TABELA_PRODUTOS = "squad1.rafael_miranda_ecommerce_produtos"
TABELA_CATEGORIAS = "squad1.rafael_miranda_ecommerce_categorias"

In [0]:
#Célula 6 — gravação em append, sem duplicar reprocessamento
def gravar_append_sem_duplicar(df, tabela, chave):
    """Grava só as linhas cuja chave ainda não existe na tabela (idempotente por _snapshot)."""
    try:
        existentes = (spark.read.format("sqlserver").options(**opts_sql())
                      .option("dbtable", tabela).load()
                      .select(*chave).distinct())
        novas = df.join(existentes, chave, "left_anti")
        n = novas.count()
    except Exception:
        novas = df  # tabela ainda não existe: primeira carga
        n = novas.count()
    if n == 0:
        print(f"{tabela}: nada novo para gravar (0 linhas)")
        return
    (novas.write.format("sqlserver").options(**opts_sql())
        .option("dbtable", tabela).mode("append").save())
    print(f"{tabela}: {n} linhas gravadas")

gravar_append_sem_duplicar(df_produtos, TABELA_PRODUTOS, ["sku", "_snapshot"])
gravar_append_sem_duplicar(df_categorias, TABELA_CATEGORIAS, ["id_categoria"])

In [0]:
#Célula 7 — conferência
for t in [TABELA_PRODUTOS, TABELA_CATEGORIAS]:
    n = spark.read.format("sqlserver").options(**opts_sql()).option("dbtable", t).load().count()
    print(f"{t}: {n} linhas na tabela")

In [0]:
# Célula 8 — métricas verticais de Data Quality
from datetime import datetime, timezone

SCHEMA_METRICAS = ("batch_id string, reference_date date, table_name string, column_name string, "
                    "metric_name string, metric_value double, total_records long, affected_records long, "
                    "quality_score double, severity string, status string, checked_at timestamp")

def _linha(batch_id, ref_date, tabela, coluna, metrica, total, afetados, severidade, agora):
    pct = round(afetados / total * 100, 2) if total else 0.0
    status = "pass" if afetados == 0 else "fail"
    return (batch_id, ref_date, tabela, coluna, metrica, pct, total, int(afetados),
            round(100 - pct, 2), severidade, status, agora)

def calcular_outliers_globais(df_todos_batches):
    """Mediana do SKU calculada entre TODOS os batches (não dentro de um só), porque
    dentro de um batch o SKU é único e a comparação ficaria sempre vazia."""
    mediana_global = (df_todos_batches.groupBy("sku")
        .agg(F.expr("percentile_approx(preco_lista, 0.5)").alias("mediana_sku_global"),
             F.count("*").alias("n_aparicoes")))
    return (df_todos_batches.join(mediana_global, "sku")
        .withColumn("_outlier", (F.col("n_aparicoes") > 1) & (F.col("mediana_sku_global") > 0) &
                    ((F.col("preco_lista") > F.col("mediana_sku_global") * 3) |
                     (F.col("preco_lista") < F.col("mediana_sku_global") * 0.3))))

def metricas_estruturais(df_batch, df_com_outlier, df_categorias, batch_id, ref_date, tabela, agora):
    total = df_batch.count()
    linhas = []
    for c in ["sku", "nome_produto", "preco_lista", "id_categoria"]:
        n = df_batch.filter(F.col(c).isNull()).count()
        linhas.append(_linha(batch_id, ref_date, tabela, c, "null_percentage", total, n, "high", agora))

    n_dup = total - df_batch.dropDuplicates(["sku"]).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "sku", "duplicate_percentage", total, n_dup, "high", agora))

    n_neg = df_batch.filter(F.col("preco_lista") <= 0).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "preco_lista", "negative_or_zero_price_pct", total, n_neg, "high", agora))

    n_orf = df_batch.join(df_categorias, "id_categoria", "left_anti").count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria", "orphan_records_pct", total, n_orf, "high", agora))

    # outlier: agora usa mediana GLOBAL do SKU (calculada fora desta função), filtrada para este batch
    n_out = df_com_outlier.filter((F.col("_batch_id") == batch_id) & F.col("_outlier")).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "preco_lista", "price_outlier_pct_exploratorio", total, n_out, "medium", agora))

    linhas.append((batch_id, ref_date, tabela, None, "row_count", float(total), total, 0, 100.0, "low", "pass", agora))
    linhas.append((batch_id, ref_date, tabela, "sku", "distinct_sku_count",
                    float(df_batch.select("sku").distinct().count()), total, 0, 100.0, "low", "pass", agora))
    return linhas

def metricas_temporais(df_todos_batches, tabela, agora):
    """Compara SKUs entre batches consecutivos: novos e desaparecidos."""
    batches = sorted(r["_batch_id"] for r in df_todos_batches.select("_batch_id").distinct().collect())
    linhas = []
    for anterior, atual in zip(batches, batches[1:]):
        sku_ant = set(r["sku"] for r in df_todos_batches.filter(F.col("_batch_id") == anterior).select("sku").collect())
        sku_atu = set(r["sku"] for r in df_todos_batches.filter(F.col("_batch_id") == atual).select("sku").collect())
        ref_date = df_todos_batches.filter(F.col("_batch_id") == atual).select("_reference_date").first()["_reference_date"]
        total = len(sku_atu)
        novos, sumidos = len(sku_atu - sku_ant), len(sku_ant - sku_atu)
        linhas.append((atual, ref_date, tabela, "sku", "new_sku_count_vs_batch_anterior",
                        float(novos), total, novos, None, "low", "info", agora))
        linhas.append((atual, ref_date, tabela, "sku", "missing_sku_count_vs_batch_anterior",
                        float(sumidos), total, sumidos, None, "low", "info", agora))
    return linhas

agora = datetime.now(timezone.utc)
df_com_outlier = calcular_outliers_globais(df_produtos)

linhas_totais = []
for b in sorted(r["_batch_id"] for r in df_produtos.select("_batch_id").distinct().collect()):
    sub = df_produtos.filter(F.col("_batch_id") == b)
    ref_date = sub.select("_reference_date").first()["_reference_date"]
    linhas_totais += metricas_estruturais(sub, df_com_outlier, df_categorias, b, ref_date, "ecommerce_produtos", agora)
linhas_totais += metricas_temporais(df_produtos, "ecommerce_produtos", agora)

df_metricas = spark.createDataFrame(linhas_totais, schema=SCHEMA_METRICAS)
df_metricas.filter(F.col("metric_name") == "price_outlier_pct_exploratorio").orderBy("batch_id").show(truncate=False)

In [0]:
# Célula 9 — gravar métricas
TABELA_METRICAS = "squad1.rafael_miranda_dq_metrics"
gravar_append_sem_duplicar(df_metricas, TABELA_METRICAS, ["batch_id", "table_name", "column_name", "metric_name"])